# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as a single object, not as a dict
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list available record set `@id`s, their fields, and columns.

In [ ]:
# List available record sets and their fields

# The Croissant schema typically defines record sets under 'recordSet', but in this schema, it appears empty.
# We'll fetch record sets directly from the dataset object.

# Find available record sets by dataset.records(record_set=...) usage.
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for fld in fields:
        if isinstance(fld, dict):
            print(f"    Field @id: {fld.get('@id')} Name: {fld.get('name')}")
        else:
            print(f"    Field @id: {fld}")
    columns = rs.get('column', [])
    if not isinstance(columns, list):
        columns = [columns]
    if columns:
        print("  Columns:")
        for col in columns:
            if isinstance(col, dict):
                print(f"    Column @id: {col.get('@id')} Name: {col.get('name')}")
            else:
                print(f"    Column @id: {col}")
    print()

# Let's preview actual records for each record set
for rs in record_sets:
    records = list(dataset.records(record_set=rs['@id']))
    print(f"Records from {rs['@id']} (showing up to 2):")
    print(records[:2])

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

We'll extract all available record sets into DataFrames.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
# For demonstration, print record set IDs
print("Available record set @ids:")
for rid in record_set_ids:
    print(f"  {rid}")

# Load all record sets
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for RecordSet {record_set_id}: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

We'll demonstrate this on one record set (e.g., the one with hormone levels, or clinical features).

In [ ]:
# Example: Use the first available record set
example_record_set_id = record_set_ids[0]
df = dataframes[example_record_set_id]

# List available fields
print(f"Fields in RecordSet {example_record_set_id}: {df.columns.tolist()}")

# Try to select a numeric field (guess, or pick one)
numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
if not numeric_fields:
    # Try to load all numeric-like columns by name (e.g. 'age_at_diagnosis', 'height', 'hormone_level')
    numeric_fields = [col for col in df.columns if 'age' in col or 'height' in col or 'level' in col or 'hormone' in col]

if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    numeric_field = df.columns[0]
print(f"Using numeric field: {numeric_field}")

threshold = 10
try:
    filtered_df = df[df[numeric_field].astype(float) > threshold]
except Exception as e:
    filtered_df = df.copy() # If not numeric, skip filtering

print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
try:
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
    ) / filtered_df[numeric_field].astype(float).std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
except Exception as e:
    print("Normalization skipped: not numeric or insufficient records.")

# Grouping: Try grouping on a categorical field
group_fields = [col for col in df.columns if df[col].dtype=='object' or 'phenotype' in col or 'zygosity' in col or 'diagnosis' in col]
if group_fields:
    group_field = group_fields[0]
    print(f"Grouping by field: {group_field}")
    try:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    except Exception as e:
        print("Grouping skipped: field not valid.")
else:
    group_field = None

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we'll plot the selected numeric field, and if grouping worked, also a grouped bar plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
if numeric_field in df.columns:
    try:
        plt.figure(figsize=(6,4))
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        sns.histplot(df[numeric_field].dropna())
        plt.title(f'{numeric_field} distribution')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()
    except Exception as e:
        print("Plotting skipped: field not numeric.")

# If grouped data available
if 'grouped_df' in locals() and grouped_df is not None and group_field is not None:
    grouped_df = grouped_df.reset_index()
    try:
        plt.figure(figsize=(8,5))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f'Average {numeric_field} by {group_field}')
        plt.show()
    except Exception as e:
        print("Grouped bar plot skipped.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset contains structured clinical, hormone, imaging, and genetic records of three female patients with non-classical 21-hydroxylase deficiency-associated infertility, described and managed in Lanzhou, China.
- Using the `mlcroissant` library, we loaded metadata and tabular data referenced via Croissant `@id` fields for reproducible access.
- Exploratory analysis demonstrates filtering and normalization on available numeric fields, as well as grouping by categorical features when present.
- Visualization provides insight into record distributions, supporting academic research and genotype-phenotype illustration.
- Due to its limited cohort (N=3), the dataset is intended for clinical illustration rather than model development.